In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
import random
from sklearn.model_selection import train_test_split

In [2]:
os.chdir('/home/ntdung/Medical')

In [6]:
excel_path = 'data/participants_1.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4920,66.0,control,f,sub-BrainAge023209,RocklandSample
4920,4921,69.0,control,m,sub-BrainAge023210,RocklandSample
4921,4922,23.0,control,m,sub-BrainAge023211,RocklandSample
4922,4923,54.0,control,f,sub-BrainAge023212,RocklandSample


In [8]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4218
Samples with decimal age values: 706


In [9]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4920,66,control,f,sub-BrainAge023209,RocklandSample
4920,4921,69,control,m,sub-BrainAge023210,RocklandSample
4921,4922,23,control,m,sub-BrainAge023211,RocklandSample
4922,4923,54,control,f,sub-BrainAge023212,RocklandSample


In [19]:
class BrainMRIDataset(Dataset):
    """
    Dataset cho Brain MRI với thông tin tuổi và giới tính thật và phản thực
    """
    def __init__(self, data_dir, participants_file, transform=None):
        self.data_dir = data_dir
        self.transform = transform

        self.participants_df = pd.read_excel(participants_file)
        self.participants_df['subject_age'] = self.participants_df['subject_age'].round().astype(int)
        
        # Chuẩn hóa giới tính thành giá trị số
        self.participants_df['gender_code'] = self.participants_df['subject_sex'].map({'m': 0, 'f': 1})
        
        # Chuẩn hóa tuổi (min-max scaling)
        self.min_age = self.participants_df['subject_age'].min()
        self.max_age = self.participants_df['subject_age'].max()
        self.age_range = self.max_age - self.min_age
        
        # Lọc các mẫu có dữ liệu MRI hợp lệ
        valid_subjects = []
        for subject_id in self.participants_df['subject_id']:
            file_path = os.path.join(data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            if os.path.exists(file_path):
                valid_subjects.append(subject_id)
        
        self.participants_df = self.participants_df[self.participants_df['subject_id'].isin(valid_subjects)]
        print(f"Tìm thấy {len(self.participants_df)} mẫu có dữ liệu MRI hợp lệ")
    
    def __len__(self):
        return len(self.participants_df)
    
    def _get_middle_slices(self, subject_id):
        try:
            file_path = os.path.join(self.data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            img = nib.load(file_path)
            img_data = img.get_fdata()
            
            # Lấy 3 lát cắt giữa
            slices = [
                img_data[:, :, img_data.shape[2]//2],  # axial
                img_data[img_data.shape[0]//2, :, :],  # sagittal
                img_data[:, img_data.shape[1]//2, :]   # coronal
            ]
            
            ## Chuyển đổi sang tensor và chuẩn hóa
            processed_slices = []
            for slice_data in slices:
                tensor_slice = torch.from_numpy(slice_data).float()
                # Min-max normalization
                if tensor_slice.max() > tensor_slice.min():  # Tránh chia cho 0
                    tensor_slice = (tensor_slice - tensor_slice.min()) / (tensor_slice.max() - tensor_slice.min())
                tensor_slice = tensor_slice.unsqueeze(0)  # Thêm chiều kênh
                if self.transform:
                    tensor_slice = self.transform(tensor_slice)
                processed_slices.append(tensor_slice)
            
            # Ghép các lát cắt thành một tensor [3, H, W]
            return torch.cat(processed_slices, dim=0)
            
        except Exception as e:
            print(f"Lỗi khi xử lý {subject_id}: {e}")
            # Trả về tensor 0 trong trường hợp lỗi
            return torch.zeros((3, 130, 130), dtype=torch.float32)
    
    def normalize_age(self, age):
        """Chuẩn hóa tuổi về khoảng [0, 1]"""
        return (age - self.min_age) / self.age_range if self.age_range > 0 else 0
    
    def __getitem__(self, idx):
        """
        Lấy một mẫu từ dataset
        """
        # Lấy thông tin mẫu thật
        real_info = self.participants_df.iloc[idx]
        real_id = real_info['subject_id']
        real_age = real_info['subject_age']
        real_gender = real_info['gender_code']
        
        # Lấy thông tin phản thực từ một mẫu ngẫu nhiên khác
        valid_indices = [i for i in range(len(self)) if i != idx]
        if not valid_indices:  # Nếu chỉ có 1 mẫu trong dataset
            cf_info = real_info
        else:
            cf_idx = random.choice(valid_indices)
            cf_info = self.participants_df.iloc[cf_idx]
        
        cf_id = cf_info['subject_id']
        cf_age = cf_info['subject_age']
        cf_gender = cf_info['gender_code']

        # Lấy ảnh và chuẩn hóa điều kiện
        real_img = self._get_middle_slices(real_id)
        
        # Chuẩn bị điều kiện
        real_condition = torch.tensor([self.normalize_age(real_age), real_gender], dtype=torch.float32)
        cf_condition = torch.tensor([self.normalize_age(cf_age), cf_gender], dtype=torch.float32)
        
        # Cũng chuẩn bị điều kiện gốc (không chuẩn hóa) cho việc kiểm tra/ghi log
        real_raw_condition = torch.tensor([real_age, real_gender], dtype=torch.float32)
        cf_raw_condition = torch.tensor([cf_age, cf_gender], dtype=torch.float32)
        
        return {
            'real_id': real_id,
            'cf_id': cf_id,
            'real_img': real_img,  # Shape: [3, H, W]
            'real_condition': real_condition,  # Shape: [2] - chuẩn hóa
            'cf_condition': cf_condition,  # Shape: [2] - chuẩn hóa
            'real_raw_condition': real_raw_condition,  # Shape: [2] - không chuẩn hóa
            'cf_raw_condition': cf_raw_condition  # Shape: [2] - không chuẩn hóa
        }

In [20]:
dataset = BrainMRIDataset(data_dir='data', participants_file='data/participants_1.xlsx')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

Tìm thấy 4924 mẫu có dữ liệu MRI hợp lệ


In [21]:
sample = dataset[0]

print("Real ID:", sample['real_id'])
print("Counterfactual ID:", sample['cf_id'])
print("Real image shape:", sample['real_img'].shape)
print("Real condition (normalized):", sample['real_condition'])
print("Counterfactual condition (normalized):", sample['cf_condition'])
print("Real condition (raw):", sample['real_raw_condition'])
print("Counterfactual condition (raw):", sample['cf_raw_condition'])

Real ID: sub-BrainAge000019
Counterfactual ID: sub-BrainAge022597
Real image shape: torch.Size([3, 130, 130])
Real condition (normalized): tensor([0.3291, 0.0000])
Counterfactual condition (normalized): tensor([0.5063, 1.0000])
Real condition (raw): tensor([44.,  0.])
Counterfactual condition (raw): tensor([58.,  1.])


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ConvBlock(nn.Module):
    """Khối tích chập cơ bản cho U-Net"""
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class SpatialTransformer(nn.Module):
    """Lớp biến đổi không gian để áp dụng biến dạng vào ảnh"""
    def __init__(self):
        super(SpatialTransformer, self).__init__()
    
    def forward(self, src, flow):
        """
        src: tensor hình ảnh nguồn [B, C, H, W]
        flow: tensor trường vector biến dạng [B, 2, H, W]
        """
        # Tạo lưới tọa độ chuẩn
        B, C, H, W = src.size()
        
        # Tạo lưới tọa độ chuẩn (từ -1 đến 1)
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, H, device=src.device),
            torch.linspace(-1, 1, W, device=src.device),
            indexing='ij'  # Thêm indexing='ij' để tương thích với PyTorch mới
        )
        grid = torch.stack((grid_x, grid_y), dim=2).unsqueeze(0)
        grid = grid.expand(B, H, W, 2)
        
        # Chuyển đổi flow từ pixel sang giá trị chuẩn hóa (-1 đến 1)
        flow_grid = flow.permute(0, 2, 3, 1)  # [B, H, W, 2]
        flow_grid = flow_grid / torch.tensor([W/2, H/2], device=flow.device)
        
        # Cộng flow vào lưới cơ sở để tạo ra lưới biến dạng
        sample_grid = grid + flow_grid
        
        # Áp dụng lưới biến dạng để lấy mẫu từ ảnh gốc
        output = F.grid_sample(src, sample_grid, mode='bilinear', padding_mode='border', align_corners=True)
        
        return output

class ScalingAndSquaring(nn.Module):
    """Lớp scaling and squaring để tích hợp trường vận tốc thành trường biến dạng"""
    def __init__(self, n_steps=7):
        super(ScalingAndSquaring, self).__init__()
        self.n_steps = n_steps
        self.transformer = SpatialTransformer()
        
    def forward(self, velocity):
        """
        velocity: tensor trường vận tốc [B, 2, H, W]
        return: trường biến dạng phi(x) thông qua tích hợp scaling and squaring
        """
        # Chia nhỏ trường vận tốc
        flow = velocity / (2 ** self.n_steps)
        
        # Thực hiện scaling and squaring
        for _ in range(self.n_steps):
            flow = flow + self.transformer(flow, flow)
            
        return flow

class Generator(nn.Module):
    """Generator U-Net với scaling and squaring layers"""
    def __init__(self, in_channels=5, init_features=16):
        """
        in_channels: số kênh đầu vào (3 cho ảnh + 2 cho điều kiện)
        """
        super(Generator, self).__init__()
        
        # Encoder
        self.encoder1 = ConvBlock(in_channels, init_features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder2 = ConvBlock(init_features, init_features*2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder3 = ConvBlock(init_features*2, init_features*2)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bridge
        self.bridge = ConvBlock(init_features*2, init_features*2)
        
        # Decoder - Sử dụng Upsample thay vì ConvTranspose2d
        self.upconv3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder3 = ConvBlock(init_features*4, init_features*2)
        
        self.upconv2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder2 = ConvBlock(init_features*4, init_features)
        
        self.upconv1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features, init_features, kernel_size=3, padding=1)
        )
        self.decoder1 = ConvBlock(init_features*2, init_features)
        
        # Velocity field prediction (2 kênh cho x và y)
        self.velocity = nn.Conv2d(init_features, 2, kernel_size=3, padding=1)
        
        # Scaling and squaring layer
        self.scaling_squaring = ScalingAndSquaring(n_steps=7)
        
        # Spatial transformer để áp dụng biến dạng
        self.transformer = SpatialTransformer()
        
    def forward(self, x, condition):
        """
        x: tensor ảnh đầu vào [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        batch_size, _, H, W = x.size()
        
        # Mở rộng điều kiện thành kênh không gian
        condition = condition.view(batch_size, 2, 1, 1).expand(-1, -1, H, W)
        
        # Ghép ảnh và điều kiện
        x_in = torch.cat([x, condition], dim=1)  # [B, 5, H, W]

        # Lưu kích thước cho việc kiểm tra
        print(f"Input shape: {x_in.shape}")
        
        # Encoder
        enc1 = self.encoder1(x_in)
        enc1_pool = self.pool1(enc1)
        print(f"enc1 shape: {enc1.shape}, enc1_pool shape: {enc1_pool.shape}")
        
        enc2 = self.encoder2(enc1_pool)
        enc2_pool = self.pool2(enc2)
        print(f"enc2 shape: {enc2.shape}, enc2_pool shape: {enc2_pool.shape}")
        
        enc3 = self.encoder3(enc2_pool)
        enc3_pool = self.pool3(enc3)
        print(f"enc3 shape: {enc3.shape}, enc3_pool shape: {enc3_pool.shape}")
        
        # Bridge
        bridge = self.bridge(enc3_pool)
        print(f"bridge shape: {bridge.shape}")
        
        # Decoder với skip connections
        up3 = self.upconv3(bridge)
        print(f"up3 shape before concat: {up3.shape}")
        print(f"enc3 shape for concat: {enc3.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up3.shape[2:] != enc3.shape[2:]:
            up3 = F.interpolate(up3, size=enc3.shape[2:], mode='bilinear', align_corners=True)
            
        up3 = torch.cat([up3, enc3], dim=1)
        print(f"up3 shape after concat: {up3.shape}")
        dec3 = self.decoder3(up3)
        print(f"dec3 shape: {dec3.shape}")
        
        up2 = self.upconv2(dec3)
        print(f"up2 shape before concat: {up2.shape}")
        print(f"enc2 shape for concat: {enc2.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up2.shape[2:] != enc2.shape[2:]:
            up2 = F.interpolate(up2, size=enc2.shape[2:], mode='bilinear', align_corners=True)
            
        up2 = torch.cat([up2, enc2], dim=1)
        print(f"up2 shape after concat: {up2.shape}")
        dec2 = self.decoder2(up2)
        print(f"dec2 shape: {dec2.shape}")
        
        up1 = self.upconv1(dec2)
        print(f"up1 shape before concat: {up1.shape}")
        print(f"enc1 shape for concat: {enc1.shape}")
        
        # Đảm bảo kích thước khớp nhau
        if up1.shape[2:] != enc1.shape[2:]:
            up1 = F.interpolate(up1, size=enc1.shape[2:], mode='bilinear', align_corners=True)
            
        up1 = torch.cat([up1, enc1], dim=1)
        print(f"up1 shape after concat: {up1.shape}")
        dec1 = self.decoder1(up1)
        print(f"dec1 shape: {dec1.shape}")
        
        # Dự đoán trường vận tốc
        velocity = self.velocity(dec1)
        print(f"velocity shape: {velocity.shape}")
        
        # Tích hợp trường vận tốc để có trường biến dạng
        flow = self.scaling_squaring(velocity)
        print(f"flow shape: {flow.shape}")
        
        # Áp dụng trường biến dạng vào ảnh gốc
        deformed_image = self.transformer(x, flow)
        print(f"deformed_image shape: {deformed_image.shape}")
        
        return deformed_image, flow

class MultiSliceGenerator(nn.Module):
    """Mô hình Generator xử lý đồng thời 3 lát cắt MRI"""
    def __init__(self):
        super(MultiSliceGenerator, self).__init__()
        # Mỗi lát cắt được xử lý bởi một generator riêng
        self.generators = nn.ModuleList([
            Generator(in_channels=3+2)  # 3 channels cho ảnh + 2 cho điều kiện
            for _ in range(3)
        ])
        
    def forward(self, x, condition):
        """
        x: tensor ảnh gốc [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        outputs = []
        flows = []
        
        # Xử lý mỗi lát cắt riêng biệt
        for i in range(3):
            slice_img = x[:, i:i+1, :, :]  # Lấy 1 lát cắt [B, 1, H, W]
            
            # Nhân rộng lát cắt thành 3 kênh để phù hợp với đầu vào của Generator
            slice_img = slice_img.repeat(1, 3, 1, 1)  # [B, 3, H, W]
            
            # Đưa vào generator
            print(f"\nProcessing slice {i+1}:")
            output, flow = self.generators[i](slice_img, condition)
            
            # Lấy kênh đầu tiên của output (vì cả 3 kênh giống nhau)
            output = output[:, 0:1, :, :]  # [B, 1, H, W]
            
            outputs.append(output)
            flows.append(flow)
        
        # Ghép 3 lát cắt đã xử lý
        generated_img = torch.cat(outputs, dim=1)  # [B, 3, H, W]
        
        return generated_img, flows

In [23]:
def test_generator():
    """Hàm kiểm tra mô hình"""
    batch_size = 2
    image = torch.randn(batch_size, 3, 130, 130)  # 3 lát cắt
    condition = torch.tensor([[0.7, 1.0], [0.3, 0.0]], dtype=torch.float32)  # Điều kiện mục tiêu
    
    model = MultiSliceGenerator()
    print("Kiểm tra mô hình MultiSliceGenerator...")
    output, flows = model(image, condition)
    
    print(f"\nTổng kết:")
    print(f"Input shape: {image.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Flow shapes: {[flow.shape for flow in flows]}")
    print("Mô hình hoạt động thành công!")
    
    return output

if __name__ == "__main__":
    test_generator()

Kiểm tra mô hình MultiSliceGenerator...

Processing slice 1:
Input shape: torch.Size([2, 5, 130, 130])
enc1 shape: torch.Size([2, 16, 130, 130]), enc1_pool shape: torch.Size([2, 16, 65, 65])
enc2 shape: torch.Size([2, 32, 65, 65]), enc2_pool shape: torch.Size([2, 32, 32, 32])
enc3 shape: torch.Size([2, 32, 32, 32]), enc3_pool shape: torch.Size([2, 32, 16, 16])
bridge shape: torch.Size([2, 32, 16, 16])
up3 shape before concat: torch.Size([2, 32, 32, 32])
enc3 shape for concat: torch.Size([2, 32, 32, 32])
up3 shape after concat: torch.Size([2, 64, 32, 32])
dec3 shape: torch.Size([2, 32, 32, 32])
up2 shape before concat: torch.Size([2, 32, 64, 64])
enc2 shape for concat: torch.Size([2, 32, 65, 65])
up2 shape after concat: torch.Size([2, 64, 65, 65])
dec2 shape: torch.Size([2, 16, 65, 65])
up1 shape before concat: torch.Size([2, 16, 130, 130])
enc1 shape for concat: torch.Size([2, 16, 130, 130])
up1 shape after concat: torch.Size([2, 32, 130, 130])
dec1 shape: torch.Size([2, 16, 130, 130])

In [13]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import random

# Thiết lập seed để kết quả tái tạo được
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

os.chdir('/home/ntdung/Medical')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, condition_dim=2):
        super(Discriminator, self).__init__()
        self.condition_dim = condition_dim
        
        # Xử lý condition vector (tuổi và giới tính)
        self.condition_mapper = nn.Sequential(
            nn.Linear(condition_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128 * 128),  # Map to spatial dimension
            nn.LeakyReLU(0.2)
        )
        
        # Mạng phân biệt ảnh
        self.model = nn.Sequential(
            # Input: Batch x (in_channels+1) x 128 x 128
            nn.Conv2d(in_channels + 1, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 64 x 64 x 64
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 128 x 32 x 32
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 256 x 16 x 16
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 512 x 8 x 8
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x, condition):
        batch_size = x.size(0)
        # Xử lý điều kiện và reshape thành tensor 2D
        condition_embedding = self.condition_mapper(condition)
        condition_embedding = condition_embedding.view(batch_size, 1, 128, 128)
        
        # Ghép ảnh và điều kiện
        x = torch.cat([x, condition_embedding], dim=1)
        
        # Xử lý qua mạng phân biệt
        return self.model(x)

### Cell 5: Điều chỉnh Generator từ DiffeoGenerator ###


### Cell 6: Kiểm tra kiến trúc model ###


### Cell 7: Định nghĩa hàm huấn luyện ###


### Cell 8: Huấn luyện mô hình ###


In [7]:
class ConditionalGenerator(nn.Module):
    """Generator để tạo ảnh MRI dựa trên điều kiện tuổi và giới tính"""
    
    def __init__(self, latent_dim=100, condition_dim=2, output_channels=3):
        super(ConditionalGenerator, self).__init__()
        self.latent_dim = latent_dim
        self.condition_dim = condition_dim
        
        # Layer xử lý véc-tơ latent và condition
        self.init_size = 8  # Kích thước ban đầu sau khi xử lý latent vector
        self.l1 = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, 128 * self.init_size * self.init_size)
        )
        
        # Mạng tạo sinh ảnh
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),  # 16x16
            
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2),  # 32x32
            
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2),  # 64x64
            
            nn.Conv2d(64, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2),  # 128x128
            
            nn.Conv2d(32, output_channels, 3, stride=1, padding=1),
            nn.Tanh()  # Chuẩn hóa đầu ra về khoảng [-1, 1]
        )
    
    def forward(self, z, condition):
        # Ghép latent vector và condition
        gen_input = torch.cat([z, condition], dim=1)
        
        # Xử lý qua fully connected layer
        out = self.l1(gen_input)
        
        # Reshape thành tensor 4D (Batch, Channels, H, W)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        
        # Xử lý qua convolutional blocks
        img = self.conv_blocks(out)
        
        return img

In [8]:
def test_models():
    # Tham số
    latent_dim = 100
    condition_dim = 2
    batch_size = 4
    
    # Khởi tạo models
    generator = ConditionalGenerator(latent_dim=latent_dim, condition_dim=condition_dim).to(device)
    discriminator = Discriminator(in_channels=3, condition_dim=condition_dim).to(device)
    
    # Tạo đầu vào giả để test
    z = torch.randn(batch_size, latent_dim).to(device)
    condition = torch.randn(batch_size, condition_dim).to(device)  # Tuổi và giới tính
    
    # Test generator
    generated_images = generator(z, condition)
    print(f"Shape của ảnh được tạo: {generated_images.shape}")
    
    # Test discriminator
    disc_output = discriminator(generated_images, condition)
    print(f"Shape của đầu ra discriminator: {disc_output.shape}")
    
    return generator, discriminator

# Chạy kiểm tra model
generator, discriminator = test_models()

Shape của ảnh được tạo: torch.Size([4, 3, 128, 128])
Shape của đầu ra discriminator: torch.Size([4, 1, 5, 5])


In [9]:
def train_gan(generator, discriminator, dataloader, num_epochs=100, latent_dim=100, 
              lr_g=0.0002, lr_d=0.0002, beta1=0.5, beta2=0.999, save_interval=10):
    
    # Khởi tạo optimizers
    optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(beta1, beta2))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(beta1, beta2))
    
    # Loss function
    adversarial_loss = nn.BCELoss()
    
    # Lưu lịch sử loss để vẽ biểu đồ
    G_losses = []
    D_losses = []
    
    # Các hình ảnh được tạo ra để theo dõi quá trình huấn luyện
    fixed_noise = torch.randn(8, latent_dim, device=device)
    # Tạo các điều kiện khác nhau để theo dõi
    fixed_conditions = torch.tensor([
        [0.2, 1.0],  # Nam, 20 tuổi
        [0.4, 1.0],  # Nam, 40 tuổi
        [0.6, 1.0],  # Nam, 60 tuổi
        [0.8, 1.0],  # Nam, 80 tuổi
        [0.2, 0.0],  # Nữ, 20 tuổi
        [0.4, 0.0],  # Nữ, 40 tuổi
        [0.6, 0.0],  # Nữ, 60 tuổi
        [0.8, 0.0],  # Nữ, 80 tuổi
    ], device=device).float()
    
    print("Bắt đầu huấn luyện...")
    
    for epoch in range(num_epochs):
        # Tiến trình cho mỗi epoch
        pbar = tqdm(enumerate(dataloader), total=len(dataloader))
        
        for i, (real_images, conditions) in pbar:
            batch_size = real_images.size(0)
            
            # Chuyển dữ liệu đến thiết bị
            real_images = real_images.to(device)
            conditions = conditions.to(device)
            
            # Nhãn thật và giả
            valid = torch.ones(batch_size, 1, 1, 1, device=device)
            fake = torch.zeros(batch_size, 1, 1, 1, device=device)
            
            #------------------
            # Huấn luyện Generator
            #------------------
            optimizer_G.zero_grad()
            
            # Tạo noise vector
            z = torch.randn(batch_size, latent_dim, device=device)
            
            # Tạo ảnh giả từ Generator
            gen_images = generator(z, conditions)
            
            # Loss cho Generator
            g_loss = adversarial_loss(discriminator(gen_images, conditions), valid)
            
            g_loss.backward()
            optimizer_G.step()
            
            #------------------
            # Huấn luyện Discriminator
            #------------------
            optimizer_D.zero_grad()
            
            # Loss cho ảnh thật
            real_loss = adversarial_loss(discriminator(real_images, conditions), valid)
            
            # Loss cho ảnh giả
            fake_loss = adversarial_loss(discriminator(gen_images.detach(), conditions), fake)
            
            # Tổng loss cho Discriminator
            d_loss = (real_loss + fake_loss) / 2
            
            d_loss.backward()
            optimizer_D.step()
            
            # Cập nhật tiến trình
            pbar.set_description(
                f"[Epoch {epoch}/{num_epochs}] "
                f"[D loss: {d_loss.item():.4f}] "
                f"[G loss: {g_loss.item():.4f}]"
            )
        
        # Lưu loss sau mỗi epoch
        G_losses.append(g_loss.item())
        D_losses.append(d_loss.item())
        
        # Lưu hình ảnh mẫu sau một số epoch
        if epoch % save_interval == 0 or epoch == num_epochs - 1:
            with torch.no_grad():
                generator.eval()
                fake_samples = generator(fixed_noise, fixed_conditions)
                generator.train()
                
                # Hiển thị hình ảnh mẫu
                plt.figure(figsize=(16, 8))
                for j in range(8):
                    gender = "Nam" if fixed_conditions[j, 1] == 1.0 else "Nữ"
                    age = int(fixed_conditions[j, 0] * 100)
                    
                    plt.subplot(2, 4, j+1)
                    # Chọn lát cắt axial để hiển thị
                    plt.imshow(fake_samples[j, 0].cpu().numpy(), cmap='gray')
                    plt.title(f"{gender}, {age} tuổi")
                    plt.axis('off')
                
                plt.tight_layout()
                plt.suptitle(f'Mẫu được tạo ở Epoch {epoch}')
                plt.savefig(f'samples_epoch_{epoch}.png')
                plt.close()
                
                # Lưu model
                torch.save({
                    'generator': generator.state_dict(),
                    'discriminator': discriminator.state_dict(),
                    'optimizer_G': optimizer_G.state_dict(),
                    'optimizer_D': optimizer_D.state_dict(),
                    'epoch': epoch
                }, f'model_checkpoint_epoch_{epoch}.pth')
    
    # Vẽ biểu đồ loss
    plt.figure(figsize=(10, 5))
    plt.title("Generator and Discriminator Loss During Training")
    plt.plot(G_losses, label="G")
    plt.plot(D_losses, label="D")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig('loss_plot.png')
    plt.close()
    
    return generator, discriminator, G_losses, D_losses

In [11]:
# Các tham số cho quá trình huấn luyện
latent_dim = 100
condition_dim = 2
batch_size = 16
num_epochs = 100
lr_g = 0.0002
lr_d = 0.0002

# Tạo lại dataset và dataloader
try:
    dataset = BrainMRIDataset(data_dir, participants_file)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    # Khởi tạo models
    generator = ConditionalGenerator(latent_dim=latent_dim, condition_dim=condition_dim).to(device)
    discriminator = Discriminator(in_channels=3, condition_dim=condition_dim).to(device)
    
    # Huấn luyện mô hình
    generator, discriminator, G_losses, D_losses = train_gan(
        generator, 
        discriminator, 
        dataloader,
        num_epochs=num_epochs,
        latent_dim=latent_dim,
        lr_g=lr_g,
        lr_d=lr_d
    )
    
    print("Hoàn thành huấn luyện!")
except Exception as e:
    print(f"Lỗi khi huấn luyện mô hình: {e}")
    print("Bạn có thể chạy cell huấn luyện khi đã chuẩn bị dữ liệu đầy đủ.")

### Cell 9: Tạo ảnh từ Generator đã huấn luyện ###


### Cell 10: Lưu và tải mô hình ###


Tìm thấy 4924 đối tượng hợp lệ
Bắt đầu huấn luyện...


  0%|          | 0/308 [00:00<?, ?it/s]


Lỗi khi huấn luyện mô hình: Using a target size (torch.Size([16, 1, 1, 1])) that is different to the input size (torch.Size([16, 1, 5, 5])) is deprecated. Please ensure they have the same size.
Bạn có thể chạy cell huấn luyện khi đã chuẩn bị dữ liệu đầy đủ.


In [ ]:
def generate_brain_images(generator, age, gender, latent_dim=100, num_samples=5):
    """
    Tạo ảnh não dựa trên tuổi và giới tính
    
    Args:
        generator: Generator đã huấn luyện
        age: Tuổi (số thực)
        gender: Giới tính ('m' hoặc 'f')
        latent_dim: Kích thước latent vector
        num_samples: Số lượng mẫu cần tạo
    """
    generator.eval()
    with torch.no_grad():
        # Tạo nhiễu ngẫu nhiên
        z = torch.randn(num_samples, latent_dim, device=device)
        
        # Chuẩn bị điều kiện
        norm_age = age / 100.0  # Chuẩn hóa tuổi
        gender_val = 1.0 if gender == 'm' else 0.0
        condition = torch.tensor([[norm_age, gender_val]]).float().to(device)
        condition = condition.repeat(num_samples, 1)
        
        # Tạo ảnh
        generated_images = generator(z, condition)
        
        # Hiển thị kết quả
        fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
        
        for i in range(num_samples):
            for j, slice_name in enumerate(['Axial', 'Coronal', 'Sagittal']):
                if num_samples > 1:
                    ax = axes[i, j]
                else:
                    ax = axes[j]
                
                # Chuyển từ [-1, 1] về [0, 1] để hiển thị
                img = (generated_images[i, j].cpu().numpy() + 1) / 2
                ax.imshow(img, cmap='gray')
                ax.set_title(f"{slice_name} Slice")
                ax.axis('off')
        
        gender_str = "Nam" if gender == 'm' else "Nữ"
        plt.suptitle(f'Ảnh MRI não được tạo: Tuổi {age}, Giới tính {gender_str}')
        plt.tight_layout()
        plt.show()

# Thử nghiệm tạo ảnh
try:
    # Tải model đã huấn luyện (nếu có)
    checkpoint_path = 'model_checkpoint_epoch_99.pth'  # Đổi tên file nếu cần
    
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        generator.load_state_dict(checkpoint['generator'])
        print("Đã tải model đã huấn luyện!")
    
    # Tạo một số ảnh với các tham số khác nhau
    print("Tạo ảnh cho nam 35 tuổi:")
    generate_brain_images(generator, age=35, gender='m', num_samples=3)
    
    print("Tạo ảnh cho nữ 70 tuổi:")
    generate_brain_images(generator, age=70, gender='f', num_samples=3)
    
except Exception as e:
    print(f"Lỗi khi tạo ảnh: {e}")
    print("Bạn cần huấn luyện mô hình trước khi tạo ảnh.")

In [ ]:
def save_model(generator, discriminator, optimizer_G, optimizer_D, epoch, filename='gan_model.pth'):
    """Lưu trạng thái mô hình GAN"""
    torch.save({
        'generator': generator.state_dict(),
        'discriminator': discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D': optimizer_D.state_dict(),
        'epoch': epoch
    }, filename)
    print(f"Đã lưu mô hình vào {filename}")

def load_model(generator, discriminator, optimizer_G=None, optimizer_D=None, filename='gan_model.pth'):
    """Tải trạng thái mô hình GAN"""
    if os.path.exists(filename):
        checkpoint = torch.load(filename)
        generator.load_state_dict(checkpoint['generator'])
        discriminator.load_state_dict(checkpoint['discriminator'])
        
        if optimizer_G is not None:
            optimizer_G.load_state_dict(checkpoint['optimizer_G'])
        if optimizer_D is not None:
            optimizer_D.load_state_dict(checkpoint['optimizer_D'])
            
        epoch = checkpoint['epoch']
        print(f"Đã tải mô hình từ epoch {epoch}")
        return epoch
    else:
        print(f"Không tìm thấy file {filename}")
        return 0

# Ví dụ cách sử dụng
try:
    # Khởi tạo optimizer mới (chỉ để minh họa)
    optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))
    
    # Lưu mô hình
    save_model(generator, discriminator, optimizer_G, optimizer_D, num_epochs, 'gan_model_final.pth')
    
    # Tải mô hình (chỉ để minh họa)
    start_epoch = load_model(generator, discriminator, optimizer_G, optimizer_D, 'gan_model_final.pth')
    print(f"Tiếp tục huấn luyện từ epoch {start_epoch}")
    
except Exception as e:
    print(f"Lỗi khi lưu/tải mô hình: {e}")

print("Hoàn thành code cho dự án GAN sinh ảnh não dựa trên điều kiện tuổi và giới tính!")